## 1: setup

In [7]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD

%load_ext autoreload
%autoreload 2
from src.evaluation import evaluate

train = pd.read_csv("../data/processed/train.csv")
val = pd.read_csv("../data/processed/val.csv")
movies = pd.read_csv("../data/ml-latest-small/movies.csv")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2: the matrix

In [8]:
user_ids = np.sort(train["userId"].unique())
item_ids = np.sort(train["movieId"].unique())
user_index = {u: i for i, u in enumerate(user_ids)}
item_index = {m: i for i, m in enumerate(item_ids)}

rows = train["userId"].map(user_index)
cols = train["movieId"].map(item_index)
R = csr_matrix((train["rating"], (rows, cols)), shape=(len(user_ids), len(item_ids)))

print("Matrix shape (users x movies):", R.shape)
print("Ratings stored:", R.nnz)

Matrix shape (users x movies): (602, 3637)
Ratings stored: 63357


## 3: a reusable recommender builder

In [9]:
def make_recommender(scores):
    def recommend(user_id, seen, k):
        u = user_index[user_id]
        user_scores = scores[u].copy()
        seen_idx = [item_index[m] for m in seen if m in item_index]
        user_scores[seen_idx] = -np.inf
        top = np.argpartition(-user_scores, k)[:k]
        top = top[np.argsort(-user_scores[top])]
        return item_ids[top].tolist()
    return recommend

## 4: try several sizes and compare on validation

In [10]:
def run_svd(n):
    svd = TruncatedSVD(n_components=n, random_state=42)
    user_factors = svd.fit_transform(R)
    item_factors = svd.components_.T
    scores_svd = user_factors @ item_factors.T

    r = evaluate(make_recommender(scores_svd), history=train, target=val, k=10)
    r["n_components"] = n
    r["coverage"] = r["n_unique_items_recommended"] / len(item_ids)
    return r

results = [run_svd(n) for n in [2, 5, 10, 15, 20, 50, 100]]
table = pd.DataFrame(results).set_index("n_components")[
    ["precision@k", "recall@k", "ndcg@k", "coverage"]
].round(4)
print(table.to_string())

              precision@k  recall@k  ndcg@k  coverage
n_components                                         
2                  0.0580    0.0721  0.0828    0.0374
5                  0.0710    0.0976  0.1034    0.0726
10                 0.0822    0.1095  0.1172    0.0965
15                 0.0816    0.1109  0.1166    0.1202
20                 0.0787    0.1100  0.1139    0.1325
50                 0.0741    0.1108  0.1089    0.1740
100                0.0638    0.1068  0.0957    0.2139


#### Findings: SVD matrix factorization (validation set, k=10)

I tuned the number of components on the validation set. NDCG@10 rises from 0.0828 (2 components) to a peak of about 0.117 at 10 to 15 components, then falls to 0.0957 at 100, which suggests overfitting in a sparse dataset. Coverage rises steadily with size, from 3.7% to 21.4%. Very small models behave like the most-popular baseline.

I chose 15 components: its accuracy is within noise of the best (NDCG@10 0.1166 vs 0.1172 at 10) and coverage is higher (12.0% vs 9.7%). It beats item-kNN by about 15% on precision, 9% on recall and 17% on NDCG at similar coverage.

Limitation: TruncatedSVD treats unrated cells as zeros rather than as missing. A method fitted only on observed ratings (ALS, for example) is a candidate improvement.